# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-37: Pulse-rate variability on VitalVideos-WW - the confirming run

### What this notebook does
Tests one finding under a rule fixed before any camera beat was computed on this set: whether the 60 s SDNN that PhysNet v2 recovers from video, on recordings the app's own gates accept, lands within 10% of a contact PPG's often enough to clear a bar set in advance - and whether, against ECG, it earns the word "HRV" or stays pulse-rate variability (PRV). Part 0 is that rule, committed before the run, with the facts it rests on and a freeze record that checks the code against it. The run itself (stage 2) scores every estimator once, through `app/live_vitals/scripts/check_vv_hrv.py`; the later sections verify its provenance, apply the rule and state the outcome. No frame is rendered, and no recording appears beside its skin type, site or age.

### Estimators
- `hr_physnet_v2` - the shipped model (NB_P3_18, configuration `baseline`, epoch 27; weights SHA-256 f30fae25...e6b8). The finding is about this estimator.
- `hr_spectral` (POS) - the classical control, its trace inverted as the harness inverts it.
- `hr_physnet_v3` - a candidate (NB_P3_31 arm A, seed 42), scored only on the 60 recordings of VitalVideos-WW's validation split: it trained on the other 240.

Three columns, never pooled.

### Set and references
VitalVideos-WW: 300 participants, one 60 s recording each, seated and still, breathing spontaneously; lossless lab video at 60 fps, stored as 72x72 face crops, 8 bits per channel, at 30 fps (every second frame). Two references on one device clock: a 500 Hz ECG, whose R-R intervals are the primary reference, and a 500 Hz contact PPG, the PRV reference and the floor. A finger oximeter (`cms_ppg`, 59.55 Hz, the model the Rice set used) is reported beside them and decides nothing.

### Background
On the Rice iPPG set (14 subjects, one camera, the app's gates bypassed), `check_rice_hrv.py` found v2's SDNN within 10% of a finger oximeter in 8 of 12 stationary recordings (median error +2.6%), and RMSSD out of reach (5 of 12). Those are discovery figures - the condition, the 60 s unit and the level breakdown were chosen on them - so they stay exploratory. VitalVideos-WW is the confirming set: an independent camera, an ECG, and the one HRV-capable set the app's gates mostly accept. A camera-free first stage (2026-10-30) settled the references, the accepted set and the floor before the rule was frozen; Part 0 records its results as facts.

## Part 0 - Pre-registration (2026-10-30)

Fixed after a camera-free first stage and before any v2, v3 or POS beat was computed on VitalVideos-WW, and committed before the run. Stage 2 runs once; its code is tested on synthetic signals and on the Rice store, and no camera beat is computed on VitalVideos-WW before the run. **Anything that changes after the run starts makes it exploratory**, reported as such with the change and its reason. The decision numbers were agreed on 2026-10-29; the first stage was allowed to adjust procedure, not the bar, and changed no number.

**Question.** On recordings the app's gates accept, does PhysNet v2's 60 s SDNN land within 10% of a contact PPG's often enough to clear a bar fixed in advance - and, against ECG, does it earn the word "HRV"?

**Scope.** A finding about a method, on one dataset: seated adults at rest, breathing spontaneously, lossless lab video. It is not a capability of the app - the app's HRV tile stays "under evaluation", and an app-level claim needs its own rule and a browser capture. The quantity is named pulse-rate variability (PRV), with its reference, unless the test below earns "HRV". No clinical or physiological reading of the number (stress, recovery, fitness, autonomic balance, health status, norms, cut-offs) and no statement about heart rhythm. Skin type and site are confounded in this set (the outdoor CH* site is Fitzpatrick 4-6), so no stratum separates them. Retrospective data only; nothing trains on Rice or MR-NIRP; anything published is aggregates only.

### Data and sets
- **Recordings.** All 300 of `vitalvideos_phase3_rc_none.h5`, read-only, through its `frames` exactly as stored (1800 frames, 72x72, 8 bits, 30 fps); no other crop.
- **Accepted.** The app's capture gates, re-run on the stored fields - framing on the stored box, frame rate, motion on the stored crops, the face count - do not refuse, and v2's status is `ok`, `degraded_capture` or `unstable`. The second-face and face-outside-the-box checks need per-sample boxes the store does not keep, so they are not re-run. Stage 1: 243.
- **Primary set.** Accepted, with an ECG the store marks usable (`ecg_ok`), no ECG or cardiac note, and R-R intervals that pass the screen the finger reference had on Rice: at least 90% kept by the physiological rules below, and at least 30 kept. Stage 1: 236. POS is scored on the same recordings, and v3 on those of them in the validation split (49).
- **Reported apart, never pooled:** the 57 recordings the gates refuse, scored as forced readings; the 7 accepted recordings with a note or an ECG that fails the screen.

### References
Both at 500 Hz on the BIO device's clock. **ECG R peaks** (`check_vv_hrv.r_peaks`): QRS-band energy (5-20 Hz, integrated over 150 ms) finds each beat whatever the lead's polarity; candidates at least 0.55 of the energy's dominant period apart and at least 0.3 of the median candidate; the apex taken within 80 ms on the band-passed lead and refined with a parabola. **Contact-PPG upstrokes** (`steepest_rises`), the camera's own fiducial, so camera against contact compares timing, not two definitions of a beat. The finger oximeter's upstrokes by the same fiducial at 59.55 Hz, on its own clock, for description only.

### Procedure, frozen (the harness as it ran on Rice and MR-NIRP)
- **Trace.** PhysNet: the central 80 frames of each 160-frame window (stride 80), each piece z-scored, so the trace covers frames 40 to 1760 of 1800. POS: `pos(rgb_trace(frames))` over every frame, inverted (`INVERTED = ("hr_spectral",)`), because it runs opposite to the stored label - on VitalVideos-WW in 300 of 300 recordings.
- **Beats.** The steepest point of each upstroke: band-pass 0.7-3.5 Hz (Butterworth, order 2, zero phase); slope peaks at least 0.55 of the trace's dominant (Welch) period apart and at least 0.4 of the median slope of the nine candidates around them; each refined with a parabola through three samples.
- **Clock.** The camera's beats move onto the BIO clock by one offset per recording, found against the contact-PPG upstrokes: the offset with the smallest median distance between paired beats, searched over +-0.4 s in 2.5 ms steps, then within +-0.25 s of it. The ECG shares the clock, so the same offset serves it; the finger oximeter gets its own by the same search. A constant offset cannot change an interval.
- **Intervals.** On every side the same two rules: inside 35-180 bpm, and within 25% of the median of the seven intervals around. SDNN (sample standard deviation) and RMSSD (over successive differences whose two intervals are both kept) of the kept intervals ending inside the recording's span on the BIO clock, from 0 to its end (about 60 s).
- **Scored.** A comparison is scored where both sides keep at least 30 intervals. In the primary set the references always do (stage 1: at least 48 for the ECG, 49 for the contact PPG), so **a camera that keeps fewer counts as outside tolerance** and stays in the denominator.
- **Error.** Per recording, camera SDNN / reference SDNN - 1.
- **The edges.** PhysNet's trace leaves 1.33 s uncovered at each end of the minute; POS's covers all of it. Measured camera-free on the primary set (2026-10-30), the references' own SDNN over the covered seconds differs from their whole-minute SDNN by more than 10% in 0.4-0.8% of recordings for the contact PPG and 2-3% for the ECG (median about +0.2%). It is part of the method's 60 s SDNN, as it was on Rice, and is not corrected.

### The rule
- **Measure.** Per recording, the absolute relative error of the 60 s SDNN. **Tolerance 10%**; 20% is reported beside it and decides nothing.
- **Share and interval.** The share of the primary set within tolerance, with a two-sided 95% Wilson interval (z = 1.959964).
- **The bar and its three outcomes**, for the pooled set and every stratum alike: *confirmed* when the lower bound is at least 50%; *not confirmed* when the upper bound is below 50%; *inconclusive* between - reported as such, never reframed. At 236 recordings 134 confirm (56.8%) and 102 or fewer are not confirmed (43.2%); a 55% share carries 48.7-61.3%.
- **Applied three times, for v2:** camera against the contact PPG (the PRV sentence); the contact PPG against ECG (the floor); camera against ECG. If the PRV bar is not confirmed, the finding is labelled Rice's rig only, with VitalVideos-WW's figures as the negative.
- **The word "HRV"** is considered only once the PRV bar is confirmed, and needs all three: **(1)** the contact PPG confirmed against ECG; **(2)** v2 confirmed against ECG; **(3)** on the primary recordings, v2's share against ECG no more than 10 points below the contact PPG's - the 95% lower bound of the paired difference (v2 minus contact PPG) at least -10 points. Otherwise the sentence says PRV against the contact PPG, an acceptable outcome. Either way it quotes both sensors' figures, not only the verdict.
- **Paired differences.** Recordings resampled with replacement (one recording per participant), 10,000 draws, seed 42 (`numpy.random.default_rng`), the 2.5th and 97.5th percentiles. Decisive only for condition 3; v2 minus POS and, on the validation split, v3 minus v2 are reported from the same procedure.
- **Strata, fixed now.** Fitzpatrick 1-2, 3-4, 5-6; site group CH*, CRC/SCH/BET, MOL/RING; age up to 40, 41-60, over 60; terciles of the ECG's SDNN (the level axis), cut at 17.83 and 29.73 ms; terciles of `qc_pos_bvp_r` (a signal quality that does not depend on the outcome), cut at -0.848 and -0.759. A value on a cut joins the lower tercile; the exact cuts are in the freeze record. Each stratum gets its own outcome for v2 against the contact PPG; the same table against ECG is description. The pooled set decides; a stratum that is not confirmed is named in the sentence, and the inconclusive ones are counted. With 15 strata one is named by chance roughly a quarter to a third of the time: **a named stratum is a flag, not a finding.**
- **RMSSD** stays a tested negative for this phase. The trigger for reopening it and the parked region-fusion rule: v2's RMSSD within 10% of the contact PPG's in at least one primary recording in four (point estimate). Below it, RMSSD is closed as a published negative on this rig; at or above it, the fusion rule is redesigned before anything runs, with its criterion on a new capture.

### Hypotheses, stated in advance
Judged on point estimates, for v2 against the contact PPG unless stated; the intervals are reported beside them.
- **H1.** v2's share within 10% is 45-65%. Contradicted outside.
- **H2** (2026-10-29). The contact PPG does not clear the bar against ECG, so "HRV" is withheld (low confidence). *Answered before the freeze, so not a test:* stage 1 contradicted it on 2026-10-30 (93.6% within 10%, 95% 89.8-96.1%).
- **H2b** (restated 2026-10-30, after stage 1 and before any camera beat). "HRV" stays withheld. Contradicted if the word is earned. High confidence: with the contact PPG at 93.6%, condition 3 needs v2 at about 87-89% within 10% of ECG (simulated on stage 1's figures), against H1's 45-65%.
- **H3.** v2's share in the lowest ECG-SDNN tercile is at least 15 points below the highest's (Rice, stationary: 50% and 83%).
- **H4.** The outdoor CH* site is v2's weakest site stratum, at least 10 points below the best. The gates have already removed the worst recordings, so the 39-point gap in heart-rate coverage between sites is not expected here.
- **H5.** v2's RMSSD is within 10% of the contact PPG's in 15-25% of the primary set, just under the trigger (low confidence).
- **H6.** v2's share exceeds POS's by 5-15 points (Rice, stationary: 67% and 50%).

### Reported, deciding nothing
- **POS and v3**, as columns beside v2: their shares, intervals and outcomes, never substituted for v2's. v3's 49 recordings confirm at 32 (65.3%); a 55% share there carries 41-68%.
- **For every estimator:** the 20% tolerance, the median error and its interquartile range, and RMSSD the same way.
- **Timing against the contact PPG:** the share of its upstrokes with a camera beat within 150 ms, the median distance of those, and the jitter each excess implies (the square root of (camera² - reference²) / k, k = 2 for SDNN and 6 for RMSSD), read as a bracket and never converted from one statistic to the other.
- **The finger oximeter:** v2's SDNN against it, recordings with `note_cms_bad` left out - the reference type the Rice figures used.
- **Three untested statistics**, descriptive only; promoting any of them to a candidate needs its own rule and a fresh set. *pNN50:* the share of successive differences over 50 ms, among pairs of kept intervals; camera minus reference, in points. *HF-band power:* the Lomb-Scargle power of the kept intervals at their end times, mean removed, summed over 0.15-0.40 Hz on a 0.005 Hz grid; relative error. *Breathing-band amplitude:* in recordings with `resp_ok`, the amplitude of a sinusoid at the belt's rate (`rr_belt`) fitted with a constant to the kept intervals by least squares; relative error. Each as the median error and its interquartile range over the primary set.
- **Forced readings and flagged recordings**, scored the same way, each group on its own.
- Every estimator's beat times are saved beside the outputs (not committed) for the after-items.

### The sentence
Its wording is fixed now; the run fills the slots, and the verb follows mechanically from the outcomes. Model card (the README takes the first clause and a link):

> Pulse-rate variability (60 s SDNN), stationary adults with normal breathing. On the [n] of 300 VitalVideos-WW recordings the app's gates accept with a usable, unflagged ECG (lossless lab video at 30 fps, not a browser capture), PhysNet v2 [recovers / was neither shown to recover nor shown not to / does not recover] 60 s SDNN within 10% of a contact PPG in [share]% of recordings (95% interval [lo-hi]; within 20%: [..]%). Against ECG the contact PPG itself lands within 10% in [f]% and the camera in [c]%, so the word "HRV" [is / is not] earned. POS: [x]%; v3 on the [m] of its 60 validation recordings among them: [y]%. Strata not confirmed: [list]; [k] inconclusive. RMSSD is a negative on this rig: [r]% within 10%. Exploratory and separate: on Rice (14 subjects, one camera, gates bypassed; 12 stationary recordings), v2's SDNN had a +2.6% median error with 67% within 10% of a finger oximeter (58-75% on 60 s spans). Research use only; no clinical, health-state or rhythm interpretation.

The slots: [n] the primary set; the verb *recovers*, *was neither shown to recover nor shown not to* or *does not recover* for confirmed, inconclusive or not confirmed; [share] and [lo-hi] v2 against the contact PPG, with its Wilson interval; [f] the contact PPG and [c] v2 against ECG; "is" only when all three conditions pass; [x] POS on the primary set; [m] and [y] v3 on the primary recordings of the validation split; [list] and [k] the strata of v2 against the contact PPG; [r] v2's RMSSD against the contact PPG. Never dropped: the quantity's name, the condition, n and coverage, the share and interval at 10% and 20%, the contact sensor's equivalent, the POS and v3 columns, the named strata with the count of inconclusive ones, the RMSSD line, the Rice line labelled exploratory, the lossless-video caveat, the research-use line, and (model card only) the weights hash and date.

### Preflight
The stage-2 outputs are scored only if every check passes; any failure stops the notebook.
1. Every output names one commit with no uncommitted app code, and between Part 0's commit and that commit nothing under `app/` changes except `check_vv_hrv.py` and its test files.
2. The registered weights: v2 f30fae25...e6b8, v3 3fab82a8...0600.
3. No `--limit`: v2 and POS on all 300 recordings, v3 on the 60 of the validation split, every estimator on the same recordings.
4. The reference side reproduces stage 1: the same primary set (its fingerprint in the freeze record), the same funnel, the same floor.
5. The settings the run records equal the freeze record's.
6. The self-test row comes first: the contact PPG scored as a camera against itself reads 0.00 median error, 100% within 10% and 0.0 ms.

### Interpretation fixed in advance
A confirmed result means that on this dataset's lossless recordings, through the app's gates and its stored crop, v2's 60 s SDNN lands within 10% of a contact PPG's often enough to clear the bar. It does not mean the app measures HRV or PRV: a browser capture is compressed, these participants sat still, and an app-level claim needs its own rule. After stage 2, not before: the re-encode check on the 12 raw VitalVideos-WW recordings, the RMSSD trigger's verdict, and the decision on a new capture.

### On record before the freeze
**Stage 1 (2026-10-30, camera-free; `Data/vv_hrv_eval/summary_stage1.json`, run on commit f07611e with a clean tree).** No v2, v3 or POS beat was computed; the camera entered only through the capture gates and v2's heart-rate statuses, which carry no beat time.
- *Funnel.* The capture gates pass 266 (framing refuses 33, the face count 1, rate and motion none); v2 reports 276 (252 `ok`, 24 `degraded_capture`; 24 `insufficient_quality`); accepted 243; no ECG or cardiac note 238; ECG screen passed: **primary 236**.
- *The ECG reference.* Usable in 295 of 300; 4 leads run negative. Its R-peak heart rate sits a median 0.06 bpm from the audit's beat-based rate (98% within 1 bpm; 5 more than 2 bpm apart). In 4 of those 5 it matches the contact PPG's rate within 0.3 bpm, so the audit is the outlier; in the fifth the detector's spacing admits extra beats, and the screen drops that recording (43% of its intervals kept).
- *Pairing.* A median 100% of R peaks have a contact-PPG upstroke 80-450 ms later (10th percentile 98.5%; 297 of 300 recordings at 95% or more); the transit is 248 ms in the median recording.
- *The floor*, contact PPG against ECG on the primary set (reference SDNN median 23.8 ms): SDNN within 10% in **93.6%** (95% 89.8-96.1%, confirmed; median +1.4%; within 20% 97%); RMSSD in **79.7%** (74.1-84.3%, confirmed; median +1.1%; within 20% 91%). Condition 1 is met and H2 contradicted. Of the 7 recordings off by more than 20%, 4 have an ECG SDNN of 10.7-20.0 ms and 3 sit at 24-32 ms.
- *Self-test.* The ECG against itself: 100% within 10%, median +0.0%.
- *Strata on the primary set.* Fitzpatrick 1-2 / 3-4 / 5-6: 78 / 79 / 79. Sites CH* / CRC-SCH-BET / MOL-RING: 70 / 83 / 83. Age up to 40 / 41-60 / over 60: 79 / 110 / 47. ECG-SDNN terciles 79 / 78 / 79; `qc_pos_bvp_r` terciles 79 / 78 / 79.
- *POS's sign.* `qc_pos_bvp_r` is negative in all 300 recordings (primary set: highest -0.27, median -0.81).

**Also measured camera-free (2026-10-30):** the edges' effect (Procedure, above), and condition 3's arithmetic: with the contact PPG at 221 of 236, the paired lower bound reaches -10 points only with v2 at 206-210 of 236 (87.3-89.0%), depending on how its hits overlap the contact PPG's.

**Discovery figures, Rice iPPG** (exploratory; the gates bypassed; a finger oximeter as reference):

| SDNN within 10% (median error) | v2 | v3 | POS |
|:---|---:|---:|---:|
| stationary, whole recordings (12) | 67% (+2.6%) | 67% | 50% |
| stationary, first / last 60 s (12) | 58% / 75% | 42% / 58% | 50% / 58% |
| watching, whole recordings | 62% (+5.6%) | 50% | 12% |
| reading, whole recordings | 42% (+34%) | 25% | 50% |
| paced deep breathing, whole recordings (13) | 77% (+3.5%) | 85% (+4.7%) | 85% (+4.9%) |

The finger reference disagrees with itself between a recording's first and last minute by a median 9.5% at rest, so a 60 s SDNN is a noisy description of the person. Failures are partly subject-specific: the three stationary subjects with the lowest SDNN are +128%, +70% and +38% off, and the camera's readings for them overlap those of the recordings that pass. RMSSD: v2 within 10% in 5 of 12 stationary recordings. **MR-NIRP Indoor** (the Rice camera in a second session; descriptive): SDNN within 10% in 6 of 7 still recordings for v2 and v3, and 2 of 7 for POS.

**Seen before the freeze.** Rice's cuts by activity, 60 s span and level; MR-NIRP Indoor's descriptive run; VitalVideos-WW's heart-rate readings and gate outputs, and the accepted set's make-up; the NB_P3_27 audit's crude interval spread of the contact PPG against the ECG (within 10% in 15% of recordings, from simple peak detectors - far from the floor above, which uses refined beats and SDNN); stage 1's results; the edges' effect and condition 3's arithmetic. No camera beat has touched VitalVideos-WW.

### Freeze record
Checks that the code at this commit carries every setting written above, reads stage 1's committed summary back and verifies its provenance (commit f07611e, clean tree, v2's registered weights, all 300 recordings, no limit). It records the strata cuts at full precision, the decision numbers stage 2 implements, and a fingerprint of the primary set: the SHA-256 of its sorted recording names, so that stage 2 can show it scored the same recordings while the per-recording table itself stays local. The cell stops if any check fails.

In [5]:
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start):
    '''The first folder at or above `start` holding both app/ and Data/.'''
    for folder in [start, *start.parents]:
        if (folder / 'app').is_dir() and (folder / 'Data').is_dir():
            return folder
    raise FileNotFoundError(f'no repository root at or above {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.live_vitals import config
from app.live_vitals.scripts import check_hrv_from_video as video
from app.live_vitals.scripts import check_rice_hrv as harness
from app.live_vitals.scripts import check_vv_hrv as vv

# Part 0's values as written above. Each must equal the code at this commit, and stage 2 must
# run with exactly these.
FROZEN = {
    'beat band (Hz)': (0.7, 3.5),
    'beat spacing (share of the dominant period)': 0.55,
    'heart-rate range (bpm)': (35.0, 180.0),
    'local interval tolerance': 0.25,
    'span (s)': 60.0,
    'minimum kept intervals': 30,
    'inverted estimators': ('hr_spectral',),
    'reconstruction window, stride (frames)': (160, 80),
    'offset search (s)': 0.4,
    'offset search around the anchor (s)': 0.25,
    'beat match (s)': 0.15,
    'QRS band (Hz)': (5.0, 20.0),
    'QRS energy window (s)': 0.15,
    'R-peak spacing (share of the period)': 0.55,
    'R-peak energy share': 0.3,
    'R apex search (s)': 0.08,
    'pulse transit window (s)': (0.08, 0.45),
    'ECG screen: least kept share': 0.90,
    'tolerance': 0.10,
    'tolerance reported': 0.20,
    'bar (95% lower bound)': 0.50,
    'Wilson z': 1.959964,
}
IN_CODE = {
    'beat band (Hz)': harness.BEAT_BAND_HZ,
    'beat spacing (share of the dominant period)': harness.BEAT_SPACING,
    'heart-rate range (bpm)': (harness.HR_MIN_BPM, harness.HR_MAX_BPM),
    'local interval tolerance': harness.IBI_LOCAL_TOL,
    'span (s)': harness.SPAN_S,
    'minimum kept intervals': harness.MIN_SPAN_INTERVALS,
    'inverted estimators': harness.INVERTED,
    'reconstruction window, stride (frames)': (config.CLIP_LEN, config.WINDOW_STRIDE),
    'offset search (s)': video.MAX_LAG_S,
    'offset search around the anchor (s)': video.ANCHOR_HALF_WIDTH_S,
    'beat match (s)': video.MATCH_TOLERANCE_S,
    'QRS band (Hz)': vv.QRS_BAND_HZ,
    'QRS energy window (s)': vv.ENERGY_WINDOW_S,
    'R-peak spacing (share of the period)': vv.R_SPACING,
    'R-peak energy share': vv.R_ENERGY_SHARE,
    'R apex search (s)': vv.R_SEARCH_S,
    'pulse transit window (s)': vv.PTT_WINDOW_S,
    'ECG screen: least kept share': vv.REF_MIN_KEPT_SHARE,
    'tolerance': vv.TOLERANCE,
    'tolerance reported': vv.TOLERANCE_REPORTED,
    'bar (95% lower bound)': vv.BAR,
    'Wilson z': vv.Z95,
}
# Fixed here; stage 2 implements them.
RULE = {
    'HRV margin (paired 95% lower bound, points)': -10,
    'RMSSD trigger (share of the primary set, point estimate)': 0.25,
    'paired bootstrap draws': 10_000,
    'paired bootstrap seed': 42,
}
WEIGHTS = {'hr_physnet_v2': 'f30fae250fb948e2aaece9aae5ec2d77fcc438d7ef1abbe54ae878a7eecee6b8',
           'hr_physnet_v3': '3fab82a89f4863a70c18ddb3d8f8f7cbc0d923adf65f85b2d0aeaa56895e0600'}
STAGE1_COMMIT = 'f07611ec40f66d9ea3e95f9e841f0602d809c6c4'
STAGE1_DIR = REPO_ROOT / 'Data' / 'vv_hrv_eval'

problems = [f'{name}: Part 0 {value!r}, code {IN_CODE[name]!r}'
            for name, value in FROZEN.items() if IN_CODE[name] != value]

STAGE1 = json.loads((STAGE1_DIR / 'summary_stage1.json').read_text(encoding='utf-8'))
prov = STAGE1['provenance']
if prov['git']['commit'] != STAGE1_COMMIT or prov['git']['dirty']:
    problems.append(f"stage 1 ran on {prov['git']['commit']} (dirty: {prov['git']['dirty']})")
if prov['weights']['sha256'] != WEIGHTS['hr_physnet_v2']:
    problems.append(f"stage 1's status estimator weights {prov['weights']['sha256']}")
if STAGE1['limit'] is not None or STAGE1['n_recordings'] != 300:
    problems.append(f"stage 1: limit {STAGE1['limit']}, {STAGE1['n_recordings']} recordings")

# The per-recording table stays local (it carries skin type, site and age); only a
# fingerprint of the primary set's recording names is recorded here.
rows = pd.read_csv(STAGE1_DIR / 'stage1_recordings.csv')
primary = sorted(rows.loc[rows['primary'], 'recording'])
if len(rows) != STAGE1['n_recordings'] or len(primary) != STAGE1['funnel']['primary']:
    problems.append(f'stage 1 table: {len(rows)} rows, {len(primary)} primary')
PRIMARY_FINGERPRINT = hashlib.sha256('\n'.join(primary).encode('utf-8')).hexdigest()
STRATA_CUTS = {'ECG SDNN (ms)': STAGE1['strata']['ecg_sdnn_terciles']['cuts'],
               'qc_pos_bvp_r': STAGE1['strata']['qc_pos_bvp_r_terciles']['cuts']}
assert not problems, 'freeze record failed:\n  ' + '\n  '.join(problems)

print(f'{len(FROZEN)} settings of Part 0 match the code at this commit:')
for name, value in FROZEN.items():
    print(f'  {name:46} {value}')
print('fixed here, implemented by stage 2:')
for name, value in RULE.items():
    print(f'  {name:58} {value}')
funnel, floor = STAGE1['funnel'], STAGE1['floor']
print(f"\nstage 1: Data/vv_hrv_eval/summary_stage1.json | commit {STAGE1_COMMIT[:7]}, clean | "
      f"v2 weights {WEIGHTS['hr_physnet_v2'][:8]}...{WEIGHTS['hr_physnet_v2'][-4:]} | "
      f"written {prov['written_utc']}")
print(f"  capture gates pass {funnel['capture_not_rejected']} | v2 reports {funnel['reported_by_v2']} | "
      f"accepted {funnel['accepted']} | no ECG or cardiac note {funnel['accepted_unflagged']} | "
      f"primary {funnel['primary']}")
for stat in ('sdnn', 'rmssd'):
    m = floor[stat]
    print(f"  floor, contact PPG against ECG, {stat.upper():5} {m['within_10']:.1%} within 10% "
          f"(95% {m['within_10_low']:.1%}-{m['within_10_high']:.1%}, {m['within_10_outcome']}), "
          f"n={m['n']}")
print(f"  self-test, ECG against itself: {STAGE1['self_test']['sdnn']['within_10']:.0%} within 10%, "
      f"median {STAGE1['self_test']['sdnn']['median']:+.1%}")
print(f"  POS against the stored label: negative in {int((rows['qc_pos_bvp_r'] < 0).sum())} of {len(rows)}")
for axis, cuts in STRATA_CUTS.items():
    print(f'  tercile cuts, {axis}: {cuts[0]!r} and {cuts[1]!r}')
print(f'primary set: {len(primary)} recordings, SHA-256 of their sorted names {PRIMARY_FINGERPRINT}')
print(f'store: {vv.DEFAULT_STORE} ({vv.DEFAULT_STORE.stat().st_size:,} bytes)')

22 settings of Part 0 match the code at this commit:
  beat band (Hz)                                 (0.7, 3.5)
  beat spacing (share of the dominant period)    0.55
  heart-rate range (bpm)                         (35.0, 180.0)
  local interval tolerance                       0.25
  span (s)                                       60.0
  minimum kept intervals                         30
  inverted estimators                            ('hr_spectral',)
  reconstruction window, stride (frames)         (160, 80)
  offset search (s)                              0.4
  offset search around the anchor (s)            0.25
  beat match (s)                                 0.15
  QRS band (Hz)                                  (5.0, 20.0)
  QRS energy window (s)                          0.15
  R-peak spacing (share of the period)           0.55
  R-peak energy share                            0.3
  R apex search (s)                              0.08
  pulse transit window (s)                      

**Amendment to Part 0 (2026-10-30, before the run; presentational only).** In *The sentence*, the inconclusive verb reads *was neither shown to recover nor shown not to recover*: the frozen wording stopped at *not to*, which leaves the clause without its verb. No definition, threshold or outcome changes.

## 1. Inputs and provenance
Loads stage 2's summary and its per-recording table and applies Part 0's preflight: one commit with no uncommitted app code, and nothing under `app/` changed since Part 0's commit except `check_vv_hrv.py` and its tests; the registered weights; a complete run, with v2 and POS on all 300 recordings and v3 on the 60 of the validation split; a reference side identical to stage 1's (the primary set's fingerprint, the funnel, the floor); settings equal to the freeze record's; and the self-test reading zero, printed first. The cell stops if any check fails.

In [6]:
import subprocess

STAGE2_DIR = REPO_ROOT / 'Data' / 'vv_hrv_eval'
PART0_COMMIT = '6985df639ffff0088753fb531b19888de13f52fa'
# Under app/, only these may change between Part 0's commit and the run's.
MAY_CHANGE = {'app/live_vitals/scripts/check_vv_hrv.py',
              'app/live_vitals/tests/test_vv_reference_beats.py',
              'app/live_vitals/tests/test_vv_camera_scoring.py'}
# Where the run's summary records each of the freeze record's settings.
SETTING_KEYS = {
    'beat band (Hz)': 'beat_band_hz',
    'beat spacing (share of the dominant period)': 'beat_spacing',
    'heart-rate range (bpm)': 'hr_range_bpm',
    'local interval tolerance': 'local_interval_tolerance',
    'span (s)': 'span_s',
    'minimum kept intervals': 'min_span_intervals',
    'inverted estimators': 'inverted',
    'reconstruction window, stride (frames)': 'reconstruction_window_stride',
    'offset search (s)': 'offset_search_s',
    'offset search around the anchor (s)': 'offset_around_anchor_s',
    'beat match (s)': 'beat_match_s',
    'QRS band (Hz)': 'qrs_band_hz',
    'QRS energy window (s)': 'energy_window_s',
    'R-peak spacing (share of the period)': 'r_spacing',
    'R-peak energy share': 'r_energy_share',
    'R apex search (s)': 'r_search_s',
    'pulse transit window (s)': 'ptt_window_s',
    'ECG screen: least kept share': 'ref_min_kept_share',
    'tolerance': 'tolerance',
    'tolerance reported': 'tolerance_reported',
    'bar (95% lower bound)': 'bar_lower_bound',
    'Wilson z': 'wilson_z',
}
STAGE1_SECTIONS = ('n_recordings', 'funnel', 'capture_rejected', 'statuses', 'reference', 'floor',
                   'self_test', 'strata')


def git(*args):
    '''A read-only git command in the repository: its return code and output.'''
    done = subprocess.run(['git', *args], cwd=REPO_ROOT, capture_output=True, text=True)
    return done.returncode, done.stdout


S2 = json.loads((STAGE2_DIR / 'summary_stage2.json').read_text(encoding='utf-8'))
TABLE = pd.read_csv(STAGE2_DIR / 'stage2_recordings.csv', float_precision='round_trip')
PRIMARY = TABLE[TABLE['primary']]
prov, problems = S2['provenance'], []

# 1. One commit, no uncommitted app code, and nothing else under app/ changed since Part 0.
RUN_COMMIT = (prov.get('git') or {}).get('commit')
uncommitted = [f for f in (prov.get('git') or {}).get('changed', []) if f.startswith('app/')]
if not RUN_COMMIT or uncommitted:
    problems.append(f'run commit {RUN_COMMIT}, uncommitted app code {uncommitted}')
elif git('merge-base', '--is-ancestor', PART0_COMMIT, RUN_COMMIT)[0] != 0:
    problems.append(f'the run ({RUN_COMMIT[:7]}) does not follow Part 0 ({PART0_COMMIT[:7]})')
else:
    code, changed = git('diff', '--name-only', PART0_COMMIT, RUN_COMMIT, '--', 'app/')
    beyond = sorted(set(changed.split()) - MAY_CHANGE)
    if code != 0 or beyond:
        problems.append(f'app/ changed since Part 0 beyond check_vv_hrv.py and its tests: {beyond}')

# 2. The registered weights.
for name, found in (('hr_physnet_v2', (prov.get('weights') or {}).get('sha256')),
                    ('hr_physnet_v3', S2['candidate']['weights']['sha256'])):
    if found != WEIGHTS[name]:
        problems.append(f'{name} weights {found}, expected {WEIGHTS[name]}')

# 3. A complete run: no --limit, v2 and POS on all 300, v3 on the 60 of the validation split.
ran_v3 = TABLE['v3_n_beats'].notna()
if S2['limit'] is not None or S2['n_recordings'] != 300 or len(TABLE) != 300:
    problems.append(f"limit {S2['limit']}, {S2['n_recordings']} recordings, {len(TABLE)} rows")
if TABLE[['v2_n_beats', 'pos_n_beats']].isna().any().any():
    problems.append('v2 or POS missing on some recordings')
if (ran_v3.sum() != 60 or set(TABLE.loc[ran_v3, 'split']) != {'val'}
        or (TABLE['split'] == 'val').sum() != 60):
    problems.append(f"v3 ran on {ran_v3.sum()} recordings, splits "
                    f"{sorted(set(TABLE.loc[ran_v3, 'split']))}")

# 4. The reference side reproduces stage 1: the same primary set, funnel and floor.
fingerprint = hashlib.sha256('\n'.join(sorted(PRIMARY['recording'])).encode('utf-8')).hexdigest()
if fingerprint != PRIMARY_FINGERPRINT:
    problems.append(f'primary set of {len(PRIMARY)}, fingerprint {fingerprint[:12]}...')
differ = [key for key in STAGE1_SECTIONS
          if json.dumps(S2[key], sort_keys=True) != json.dumps(STAGE1[key], sort_keys=True)]
if differ:
    problems.append(f'stage 1 not reproduced in {differ}')

# 5. The settings the run records equal the freeze record's.
settings = S2['settings']
wrong = [name for name, key in SETTING_KEYS.items()
         if (tuple(settings[key]) if isinstance(settings.get(key), list) else settings.get(key))
         != FROZEN[name]]
if settings.get('hrv_margin') != RULE['HRV margin (paired 95% lower bound, points)'] / 100:
    wrong.append('HRV margin')
trigger = RULE['RMSSD trigger (share of the primary set, point estimate)']
if settings.get('rmssd_trigger') != trigger:
    wrong.append('RMSSD trigger')
if ((settings.get('bootstrap_draws'), settings.get('bootstrap_seed'))
        != (RULE['paired bootstrap draws'], RULE['paired bootstrap seed'])):
    wrong.append('paired bootstrap')
cuts = [settings.get('ecg_sdnn_cuts_ms'), settings.get('qc_pos_bvp_r_cuts')]
if cuts != list(STRATA_CUTS.values()):
    wrong.append('strata cuts')
if wrong:
    problems.append(f'settings differ from the freeze record: {wrong}')

# 6. The self-test: the contact PPG scored as a camera against itself reads 0.00, 100%, 0.0 ms.
SELF = S2['camera']['self_test']
self_median = SELF['ppg']['sdnn'].get('median')
self_timing = SELF['timing']['timing_ms'].get('median')
self_ok = (SELF['ppg']['sdnn'].get('within_10') == 1.0 and self_median is not None
           and abs(self_median) < 1e-9 and SELF.get('largest_error') is not None
           and SELF['largest_error'] < 1e-9 and self_timing is not None and abs(self_timing) < 1e-6)
if not self_ok:
    problems.append(f"self-test: {SELF['ppg']['sdnn'].get('within_10')} within 10%, median "
                    f"{self_median}, largest error {SELF.get('largest_error')}, "
                    f"timing {self_timing} ms")

if self_ok:
    print(f"self-test, the contact PPG scored as a camera against itself: median error "
          f"{self_median:+.2f}, {SELF['ppg']['sdnn']['within_10']:.0%} within 10%, timing "
          f"{self_timing:.1f} ms (largest error {SELF['largest_error']:.1e})")
assert not problems, 'preflight failed:\n  ' + '\n  '.join(problems)
print(f"run: commit {RUN_COMMIT[:7]}, no uncommitted app code; since Part 0 ({PART0_COMMIT[:7]}) "
      f"app/ changed only in {sorted(set(changed.split()))}")
print(f"weights: v2 {WEIGHTS['hr_physnet_v2'][:8]}..., v3 {WEIGHTS['hr_physnet_v3'][:8]}... | "
      f"{len(TABLE)} recordings, v2 and POS on all, v3 on {ran_v3.sum()} (validation split)")
print(f"reference side as stage 1: primary {len(PRIMARY)} (fingerprint matches), funnel and floor "
      f"identical | {len(FROZEN)} settings, the rule's four and the strata cuts as frozen")
print('preflight passed')

self-test, the contact PPG scored as a camera against itself: median error +0.00, 100% within 10%, timing 0.0 ms (largest error 3.1e-15)
run: commit 4159f31, no uncommitted app code; since Part 0 (6985df6) app/ changed only in ['app/live_vitals/scripts/check_vv_hrv.py', 'app/live_vitals/tests/test_vv_camera_scoring.py']
weights: v2 f30fae25..., v3 3fab82a8... | 300 recordings, v2 and POS on all, v3 on 60 (validation split)
reference side as stage 1: primary 236 (fingerprint matches), funnel and floor identical | 22 settings, the rule's four and the strata cuts as frozen
preflight passed


## 2. The rule and the outcome
The decisive figures are recomputed from the per-recording table with the run's own functions and must equal the summary's. The outcomes then follow mechanically from Part 0: the PRV bar for v2 against the contact PPG; the three conditions for the word "HRV" (the contact PPG against ECG, v2 against ECG, and the paired margin); the strata, of which a stratum not confirmed is a flag, not a finding; the RMSSD trigger; the hypotheses, judged on point estimates; and the sentence, filled from them.

In [7]:
import textwrap

C = S2['camera']
records = PRIMARY.to_dict('records')
# The decisive figures again, from the per-recording table and the run's own functions.
AGAIN = dict(prv=vv.share(records, 'v2_sdnn_error_ppg'),
             floor=vv.share(records, 'sdnn_error_ppg'),
             camera_ecg=vv.share(records, 'v2_sdnn_error_ecg'),
             margin=vv.paired_bootstrap(vv.hits(PRIMARY['v2_sdnn_error_ecg']),
                                        vv.hits(PRIMARY['sdnn_error_ppg'])))
RECORDED = dict(prv=C['columns']['v2']['ppg']['sdnn'], floor=C['floor'],
                camera_ecg=C['columns']['v2']['ecg']['sdnn'], margin=C['hrv']['margin'])
disagree = [key for key in AGAIN
            if json.dumps(AGAIN[key], sort_keys=True) != json.dumps(RECORDED[key], sort_keys=True)]
assert not disagree, f'the summary and the per-recording table disagree on {disagree}'

LABELS = {'fitzpatrick_group': 'Fitzpatrick {}', 'site_group': 'site {}', 'age_band': 'age {}',
          'ecg_sdnn_tercile': 'ECG SDNN, {} tercile',
          'qc_pos_bvp_r_tercile': 'signal quality, {} tercile'}


def pct(x, digits=1):
    return 'n/a' if x is None else f'{x:.{digits}%}'


def row(label, m):
    '''One share as the rule reads it.'''
    if not m.get('n'):
        return dict(comparison=label, n=0)
    return dict(comparison=label, n=m['n'], within_10=pct(m['within_10']),
                interval_95=f"{pct(m['within_10_low'])}-{pct(m['within_10_high'])}",
                outcome=m['within_10_outcome'], median_error=pct(m.get('median')),
                within_20=pct(m['within_20'], 0))


HRV = C['hrv']
margin = AGAIN['margin']
print(pd.DataFrame([row('PRV: v2 against the contact PPG', AGAIN['prv']),
                    row('(1) the contact PPG against ECG', AGAIN['floor']),
                    row('(2) v2 against ECG', AGAIN['camera_ecg'])]).set_index('comparison')
      .to_string())
print(f"(3) v2 minus the contact PPG, both against ECG: {margin['difference']:+.1%} (95% "
      f"{margin['low']:+.1%} to {margin['high']:+.1%}); bound {vv.HRV_MARGIN:+.0%}: "
      f"{'met' if HRV['margin_met'] else 'not met'}")
print(f'PRV bar: {AGAIN["prv"]["within_10_outcome"]} | the word "HRV": '
      f'{"earned" if HRV["earned"] else "not earned"}')

print('\nstrata, v2 against the contact PPG:')
print(pd.DataFrame([dict(row(LABELS[axis].format(level), m), stratum=LABELS[axis].format(level))
                    for axis, levels in C['strata']['ppg'].items() for level, m in levels.items()])
      .drop(columns='comparison').set_index('stratum').to_string())
NAMED = [LABELS[axis].format(level)
         for axis, level in (name.split(' ', 1) for name in C['strata_not_confirmed'])]
print(f"not confirmed: {', '.join(NAMED) or 'none'} | inconclusive: {C['strata_inconclusive']} of "
      f"{sum(len(levels) for levels in C['strata']['ppg'].values())}")

TRIGGER = C['rmssd_trigger']
print(f"\nRMSSD, v2 within 10% of the contact PPG's: {pct(TRIGGER['share'])} of the primary set; "
      f"trigger at {TRIGGER['threshold']:.0%}: "
      f"{'met - RMSSD and region fusion reopen' if TRIGGER['met'] else 'not met - RMSSD closes'}")

H = C['hypotheses']


def shown(name, h):
    if 'value' not in h:
        return 'n/a'
    if name == 'H2b':
        return '"HRV" earned' if h['value'] else '"HRV" not earned'
    return pct(h['value']) if name in ('H1', 'H5') else f"{h['value'] * 100:+.1f} points"


hypotheses = [dict(hypothesis='H2', prediction='the contact PPG does not clear the bar against ECG',
                   value=pct(STAGE1['floor']['sdnn']['within_10']),
                   verdict='contradicted by stage 1, before the freeze - not a test')]
hypotheses += [dict(hypothesis=name, prediction=h.get('prediction', ''), value=shown(name, h),
                    verdict=h['verdict']) for name, h in H.items()]
order = ['H1', 'H2', 'H2b', 'H3', 'H4', 'H5', 'H6']
print('\n' + pd.DataFrame(hypotheses).set_index('hypothesis').loc[order].to_string())

VERB = {'confirmed': 'recovers',
        'inconclusive': 'was neither shown to recover nor shown not to recover',
        'not confirmed': 'does not recover'}
prv, pos, v3 = AGAIN['prv'], C['columns']['pos']['ppg']['sdnn'], C['columns']['v3']['ppg']['sdnn']
SENTENCE = (
    'Pulse-rate variability (60 s SDNN), stationary adults with normal breathing. '
    f"On the {prv['n']} of 300 VitalVideos-WW recordings the app's gates accept with a usable, "
    'unflagged ECG (lossless lab video at 30 fps, not a browser capture), PhysNet v2 '
    f"{VERB[prv['within_10_outcome']]} 60 s SDNN within 10% of a contact PPG in "
    f"{prv['within_10']:.0%} of recordings (95% interval {prv['within_10_low'] * 100:.0f}-"
    f"{prv['within_10_high'] * 100:.0f}%; within 20%: {prv['within_20']:.0%}). Against ECG the "
    f"contact PPG itself lands within 10% in {AGAIN['floor']['within_10']:.0%} and the camera in "
    f"{AGAIN['camera_ecg']['within_10']:.0%}, so the word \"HRV\" "
    f"{'is' if HRV['earned'] else 'is not'} earned. POS: {pos['within_10']:.0%}; v3 on the "
    f"{v3['n']} of its 60 validation recordings among them: {v3['within_10']:.0%}. Strata not "
    f"confirmed: {', '.join(NAMED) or 'none'}; {C['strata_inconclusive']} inconclusive. RMSSD is "
    f"a negative on this rig: {TRIGGER['share']:.0%} within 10%. Exploratory and separate: on Rice "
    '(14 subjects, one camera, gates bypassed; 12 stationary recordings), v2\'s SDNN had a +2.6% '
    'median error with 67% within 10% of a finger oximeter (58-75% on 60 s spans). Research use '
    'only; no clinical, health-state or rhythm interpretation.')
print('\nthe sentence (model card; the README takes the first clause and a link):\n')
print(textwrap.fill(SENTENCE, 100))

                                   n within_10  interval_95        outcome median_error within_20
comparison                                                                                       
PRV: v2 against the contact PPG  236      9.3%   6.2%-13.7%  not confirmed        89.7%       18%
(1) the contact PPG against ECG  236     93.6%  89.8%-96.1%      confirmed         1.4%       97%
(2) v2 against ECG               236      7.6%   4.9%-11.7%  not confirmed        95.3%       17%
(3) v2 minus the contact PPG, both against ECG: -86.0% (95% -90.3% to -81.4%); bound -10%: not met
PRV bar: not confirmed | the word "HRV": not earned

strata, v2 against the contact PPG:
                                   n within_10  interval_95        outcome median_error within_20
stratum                                                                                          
Fitzpatrick 1-2                   78     20.5%  13.0%-30.8%  not confirmed        26.9%       40%
Fitzpatrick 3-4             

## 3. Descriptive results
Reported beside the rule and deciding nothing: POS and v3 as columns beside v2, and the paired differences between them; RMSSD; timing against the contact PPG and the jitter each excess implies, read as a bracket; the finger oximeter, with the contact PPG's own agreement with it; the three untested statistics; the strata against ECG; and the forced readings and flagged recordings, each on its own. Aggregates only: no recording appears beside its skin type, site or age.

In [8]:
NAMES = {'v2': 'PhysNet v2', 'pos': 'POS', 'v3': 'PhysNet v3 (validation split)'}


def short(m):
    '''A share, its interval and outcome in one cell.'''
    if not m.get('n'):
        return 'n/a'
    return (f"{pct(m['within_10'])} ({pct(m['within_10_low'], 0)}-{pct(m['within_10_high'], 0)}, "
            f"{m['within_10_outcome']})")


def spread(m, scale=1.0, unit=''):
    '''A median with its interquartile range.'''
    if not m.get('n'):
        return 'n/a'
    return f"{m['median'] * scale:.1f}{unit} ({m['q1'] * scale:.1f} to {m['q3'] * scale:.1f})"


with pd.option_context('display.width', 250, 'display.max_columns', 20,
                       'display.max_colwidth', 60):
    print('each estimator over its primary recordings (SDNN and RMSSD within 10%):')
    print(pd.DataFrame([dict(estimator=NAMES[k], n=col['n'],
                             sdnn_contact_ppg=short(col['ppg']['sdnn']),
                             median_error=pct(col['ppg']['sdnn'].get('median')),
                             sdnn_ecg=short(col['ecg']['sdnn']),
                             rmssd_contact_ppg=short(col['ppg']['rmssd']),
                             no_sdnn=col['unscored'])
                        for k, col in C['columns'].items()]).set_index('estimator').to_string())
    for label, key in (('v2 minus POS', 'v2_minus_pos'),
                       ('v3 minus v2, validation split', 'v3_minus_v2')):
        d = C['paired'][key]
        print(f"{label}, SDNN within 10% of the contact PPG: {d['difference']:+.1%} "
              f"(95% {d['low']:+.1%} to {d['high']:+.1%}, n={d['n']})"
              if d.get('n') else f'{label}: n/a')

    print('\ntiming against the contact PPG (median recording, interquartile range):')
    print(pd.DataFrame([dict(estimator=NAMES[k],
                             upstrokes_matched=spread(col['timing']['matched'], 100.0, '%'),
                             timing=spread(col['timing']['timing_ms'], unit=' ms'),
                             jitter_from_sdnn=spread(col['timing']['jitter_sdnn_ms'], unit=' ms'),
                             jitter_from_rmssd=spread(col['timing']['jitter_rmssd_ms'], unit=' ms'))
                        for k, col in C['columns'].items()]).set_index('estimator').to_string())

    print('\nthe finger oximeter (description only):')
    print(f"  v2 SDNN within 10% of the oximeter's: {short(C['columns']['v2']['oximeter'])}")
    print(f"  the contact PPG within 10% of the oximeter's: {short(C['self_test']['oximeter'])}")

    print('\nuntested statistics, camera against reference (median, interquartile range):')
    UNTESTED = {'pnn50_contact_ppg': ('pnn50_diff_ppg', 1.0, ' pt'),
                'pnn50_ecg': ('pnn50_diff_ecg', 1.0, ' pt'),
                'hf_power_contact_ppg': ('hf_error_ppg', 100.0, '%'),
                'hf_power_ecg': ('hf_error_ecg', 100.0, '%'),
                'breathing_band_contact_ppg': ('breath_error_ppg', 100.0, '%'),
                'breathing_band_ecg': ('breath_error_ecg', 100.0, '%')}
    print(pd.DataFrame({NAMES[k]: {label: spread(col['secondaries'][key], scale, unit)
                                   for label, (key, scale, unit) in UNTESTED.items()}
                        for k, col in C['columns'].items()}).to_string())

    print('\nstrata, v2 against ECG:')
    print(pd.DataFrame([dict(stratum=LABELS[axis].format(level), n=m.get('n', 0), sdnn_ecg=short(m))
                        for axis, levels in C['strata']['ecg'].items()
                        for level, m in levels.items()]).set_index('stratum').to_string())

    print('\napart, never pooled - forced readings (the gates refused) and flagged recordings:')
    print(pd.DataFrame([dict(group=group, estimator=NAMES[k], n=g['n'],
                             sdnn_contact_ppg=short(g['ppg']), sdnn_ecg=short(g['ecg']))
                        for group, members in C['apart'].items() for k, g in members.items()])
          .set_index(['group', 'estimator']).to_string())

each estimator over its primary recordings (SDNN and RMSSD within 10%):
                                 n               sdnn_contact_ppg median_error                       sdnn_ecg             rmssd_contact_ppg  no_sdnn
estimator                                                                                                                                           
PhysNet v2                     236   9.3% (6%-14%, not confirmed)        89.7%   7.6% (5%-12%, not confirmed)   3.0% (1%-6%, not confirmed)        1
POS                            236    2.1% (1%-5%, not confirmed)       167.4%    1.7% (1%-4%, not confirmed)   0.0% (0%-2%, not confirmed)        0
PhysNet v3 (validation split)   49  10.2% (4%-22%, not confirmed)        63.2%  12.2% (6%-24%, not confirmed)  8.2% (3%-19%, not confirmed)        0
v2 minus POS, SDNN within 10% of the contact PPG: +7.2% (95% +3.8% to +11.0%, n=236)
v3 minus v2, validation split, SDNN within 10% of the contact PPG: +6.1% (95% -2.0% to +14.3%, n=4

## 4. Discussion and conclusions

### Outcome under the rule
The pre-registered test does not confirm the finding. On the 236 primary recordings, PhysNet v2's 60 s SDNN lands within 10% of the contact PPG's in 9.3% (22 recordings; 95% 6.2-13.7%), against a bar of 50% for the interval's lower bound; the median error is +89.7%, and 18% land within 20%. Against ECG the contact PPG itself lands within 10% in 93.6% (condition 1 met), v2 in 7.6% (condition 2 not met), and v2 trails the contact PPG by 86 points (95% -90 to -81; condition 3 not met), so the word "HRV" is not earned. As Part 0 fixed in advance, the Rice result (SDNN within 10% in 8 of 12 stationary recordings) is a finding about Rice's rig only, with these figures as its negative. All 15 strata are not confirmed; none is inconclusive. RMSSD lands within 10% in 3.0%, under the 25% trigger: RMSSD closes as a negative on this rig, and the parked region-fusion rule stays parked. The run was made once, on the committed code, and the preflight shows that nothing Part 0 froze had changed.

### The columns and the hypotheses
POS reaches 2.1%, and v3 10.2% on its 49 validation recordings; v2 exceeds POS by 7.2 points (95% 3.8 to 11.0), and v3 exceeds v2 by 6.1 points on the shared 49 (95% -2.0 to 14.3). The order of the estimators matches Rice; the level does not. H1 (45-65%) and H5 (15-25%) are contradicted: both predictions came from Rice and were far too optimistic. H2b, H3, H4 and H6 hold: "HRV" stays withheld; the lowest ECG-SDNN tercile sits 19 points below the highest; the outdoor CH* site is the weakest site group, 20.5 points below the best; v2 is 7.2 points above POS.

### What the descriptive figures suggest
None of this was tested by the rule.
- **The camera finds the beats but times them loosely.** v2 has a beat within 150 ms of 92% of the contact PPG's upstrokes in the median recording, at a median 15.1 ms (interquartile 9.5-27.3 ms) - about twice the 8.1 ms of Rice's stationary recordings. The SDNN excess implies a median 26 ms of independent jitter, more than that distance: the error is heavy-tailed, the mark of occasional misplaced or missing beats that the 25% interval rule keeps.
- **The variability to be measured is small.** The median ECG SDNN is 23.8 ms. On Rice the stationary recordings that failed were the low-SDNN ones; here most recordings lie in that range (lowest tercile 1.3% within 10%), and even the highest tercile, above 29.7 ms, reaches only 20.3%.
- **Skin type and site.** Fitzpatrick 1-2: 20.5% (median error +27%); 3-4: 7.6%; 5-6: 0.0% (+221%); the outdoor CH* site: 0.0%. Skin type and site are confounded in this set, so the two cannot be told apart. `qc_pos_bvp_r` is negative throughout, so its lowest tercile is the strongest agreement between POS and the label: 13.9% within 10% there and 2.5% in the highest tercile, the direction a signal-quality axis should run.
- **Broadband error, a partly preserved breathing swing.** HF-band power is inflated (median +403%) and pNN50 by 29 points, while the breathing-band amplitude at the belt's rate is closer (median +14%, interquartile -16 to +96%). The camera's error is broadband timing noise, and the slow, breath-locked swing partly survives it. This is description: promoting it to a candidate needs its own rule and a fresh set.
- **Heart rate is not the problem.** On the recordings v2 reports as `ok`, its zero-shot heart-rate window error on this set is 0.37 bpm; the gap is between rate and beat timing.
- **The references.** The contact PPG agrees with the finger oximeter within 10% in only 64.3% of recordings, so the oximeter is a weak reference here (v2 against it: 13.6%); it decided nothing. A check made after the run, outside this notebook and without the camera, rules out the frame grid: the contact PPG resampled at the frame times and timed on the frame-index clock lands within 10% of itself at 500 Hz in 99.6% of recordings, and v2's errors are unrelated to it.

### What this cannot show
The run tests a finding; it does not establish why the finding fails. The timing, level and skin-type patterns above are associations on one dataset, with skin type and site confounded. Nothing here bears on heart rate, and nothing here is clinical.

### Conclusions
- **Not confirmed.** PhysNet v2 does not recover 60 s SDNN within 10% of a contact PPG on VitalVideos-WW: 9.3% of the 236 recordings the app's gates accept with a usable, unflagged ECG, against a pre-registered bar of 50%. The Rice result stands as a finding about one laboratory rig.
- **"HRV" is not earned; RMSSD is closed on this rig; region fusion stays parked.**
- **The app's variability display stays unavailable.**
- **After this run:** the re-encode check on the 12 raw VitalVideos-WW recordings, the decision on a new capture, respiratory rate from the saved beats against the belt, and review item 17 of `check_hrv_sampling_ceiling.py`.